In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78521)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 69895)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 58183)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 51601)

# #### Device() ####
# device = cuda:6



In [2]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:6)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

---

* 2b: normalization methods w/ NB loss
    * same as 2a but with NBLoss
    * 2a/b identical to reconstr_grid_3, but method='node'
    * 'ablation' of normalization methods used in grid 1 (libnom, vst, znorm)

In [3]:
from modules.layers import AttentionSetPooling
from modules.loss import NBLoss
from modules.model import CountsAutoencoder
from modules.norm import RawCounts, LogCounts, NBVST
from modules.train import Experiment, grid, kwarg_grid
from modules.trainers import ReconstrTrainer
from functools import partial
import torch.nn as nn

In [4]:
## Model

model_template = partial(
    CountsAutoencoder,
    dataset = dataset,
    # out_dim = num_nodes (default)
    embed_dim = 128,
    # head_dim = None (default)
    # num_heads = 1 (default)
    method = 'node',

    # layers
    # norm_class = (handled in grid), !!! 
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,
    variational=False,
    # out_module = nn.Linear (default)

    # layer params
    hidden_dims = 1,
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # kwargs
    # norm_kwargs = (handled in grid) !!!
    # pooling_kwargs = None (default)
)

norm_kwarg_grid = kwarg_grid(
    libnorm=[False,True],
    znorm=[False,True],
    learnable=[False,True],
)

model_grid = grid(
    model_template,
    norm_class = [RawCounts, LogCounts, NBVST],
    norm_kwargs = norm_kwarg_grid
)

display(norm_kwarg_grid)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    lr = 1e-3,
    # pos_keys = None (default)
    kw_keys={'x':'x', 'mu':'x_pred', 'theta':'theta'},
    metric_keys={'pred':'mu', 'target':'x'},
    loss_class = NBLoss,
    # loss_kwargs = None (default)
    # optim_class = torch.optim.Adam (default)
    # optim_kwargs = None (default)
    early_stop = True,
    # stop_metric = 'loss' (default)
    # stop_kwargs = None (default)
    trainer_norm_class=LogCounts,
    # trainer_norm_kwargs = None (default)

)

trainer_grid = grid(
    trainer_template,
)

## Experiment

expt = Experiment(
    num_trials=5, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

[{'libnorm': False, 'znorm': False, 'learnable': False},
 {'libnorm': False, 'znorm': False, 'learnable': True},
 {'libnorm': False, 'znorm': True, 'learnable': False},
 {'libnorm': False, 'znorm': True, 'learnable': True},
 {'libnorm': True, 'znorm': False, 'learnable': False},
 {'libnorm': True, 'znorm': False, 'learnable': True},
 {'libnorm': True, 'znorm': True, 'learnable': False},
 {'libnorm': True, 'znorm': True, 'learnable': True}]

24


In [5]:
expt.run_experiment(
    'grid_2b_NBreconstr_norm',
    report_metrics=['loss','rmse','mae','r2'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]